In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

###Task 3.1 — Features, split and baseline

In [ ]:

print("--- Task 3.1: Features, Split, and Baseline ---")

# 1. Load clean dataset
df = pd.read_csv("/content/appointments_clean.csv")

# 2. Define features (X) and target (y)
X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows"]].copy()
X["reminder_sent"] = df["reminder_sent"].map({"Yes": 1, "No": 0})

# One-hot encode booking channel and visit type
X = pd.concat([
    X,
    pd.get_dummies(df[["booking_channel", "visit_type"]], drop_first=True, dtype=int)
], axis=1)

y = df["no_show"]

# 3. Honest train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Print verification metrics
baseline_accuracy = (y_test == 0).mean()

print(f"Training set rows: {len(X_train)}")
print(f"Test set rows: {len(X_test)}")
print(f"No-shows in test set: {y_test.sum()}")
print(f"Baseline accuracy (predicting all Attended): {baseline_accuracy:.4f}")

--- Task 3.1: Features, Split, and Baseline ---
Training set rows: 792
Test set rows: 198
No-shows in test set: 49
Baseline accuracy (predicting all Attended): 0.7525


###Task 3.2 — Logistic regression

In [ ]:
print("--- Task 3.2: Logistic Regression & Evaluation ---")

# 1. Create a pipeline with StandardScaler and LogisticRegression
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# 2. Fit on training data
lr_pipe.fit(X_train, y_train)

# 3. Predict on test set
y_pred = lr_pipe.predict(X_test)
y_prob = lr_pipe.predict_proba(X_test)[:, 1]

# 4. Evaluate metrics
acc = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Logistic Regression Accuracy: {acc:.4f}")
print(f"Logistic Regression Recall: {recall:.4f}")
print(f"Logistic Regression ROC-AUC: {roc_auc:.4f}")

# 5. Inspect feature coefficients
coefs = lr_pipe.named_steps["model"].coef_[0]
feat_importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coefs
}).sort_values(by="Coefficient", ascending=False)

print("\nTop Feature Coefficients (Impact on No-Show):")
print(feat_importance.to_string(index=False))

--- Task 3.2: Logistic Regression & Evaluation ---
Logistic Regression Accuracy: 0.7879
Logistic Regression Recall: 0.3469
Logistic Regression ROC-AUC: 0.7885

Top Feature Coefficients (Impact on No-Show):
                Feature  Coefficient
         prior_no_shows     0.649563
              lead_days     0.614170
                fee_inr     0.453093
  booking_channel_Phone     0.260230
            distance_km     0.253398
   visit_type_Follow-up    -0.031067
booking_channel_Walk-In    -0.146647
                    age    -0.253787
          reminder_sent    -0.581332


###Task 3.3 — Decision tree and overfitting

In [ ]:
print("--- Task 3.3: Decision Trees & Overfitting ---")

# 1. Full-depth Decision Tree
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

full_train_acc = accuracy_score(y_train, dt_full.predict(X_train))
full_test_acc = accuracy_score(y_test, dt_full.predict(X_test))
full_depth = dt_full.get_depth()

print(f"Full-Depth Tree (Depth {full_depth}) -> Train Acc: {full_train_acc:.4f} | Test Acc: {full_test_acc:.4f}")

# 2. Regularized Depth-3 Decision Tree
dt_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_depth3.fit(X_train, y_train)

d3_train_acc = accuracy_score(y_train, dt_depth3.predict(X_train))
d3_test_acc = accuracy_score(y_test, dt_depth3.predict(X_test))

print(f"Depth-3 Tree -> Train Acc: {d3_train_acc:.4f} | Test Acc: {d3_test_acc:.4f}")

--- Task 3.3: Decision Trees & Overfitting ---
Full-Depth Tree (Depth 15) -> Train Acc: 1.0000 | Test Acc: 0.7020
Depth-3 Tree -> Train Acc: 0.8220 | Test Acc: 0.7778


###Task 3.4 — Random forest and a comparison table

In [ ]:
print("--- Task 3.4: Random Forest Baseline ---")

# 1. Train Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 2. Predict on test set
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

# 3. Evaluate metrics
rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print(f"Random Forest Test Accuracy: {rf_acc:.4f}")
print(f"Random Forest ROC-AUC: {rf_auc:.4f}")

# 4. Feature Importances
importances = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTop Feature Importances:")
print(importances.to_string(index=False))

--- Task 3.4: Random Forest Baseline ---
Random Forest Test Accuracy: 0.7525
Random Forest ROC-AUC: 0.6927

Top Feature Importances:
                Feature  Importance
              lead_days    0.217616
                    age    0.207938
            distance_km    0.196405
                fee_inr    0.146893
         prior_no_shows    0.102790
          reminder_sent    0.045645
  booking_channel_Phone    0.034492
   visit_type_Follow-up    0.026097
booking_channel_Walk-In    0.022123
